# V6.4 Clean — Camber Bootstrap + Post-Clean Audit

Bản này được viết cho trường hợp **reset kernel / filesystem local bị mất**.

Nó không đi tìm `/home/jovyan/.../best.pt` trước nữa. Thay vào đó:

1. Lấy `CAMBER_API_KEY` từ Kaggle Secrets.
2. Tải lại toàn bộ bundle V6.4-clean từ Camber:
   - `best.pt`
   - `data.yaml`
   - `v6p4_clean_official.freeze.json`
   - `v6p4_clean_reconciled_labels.zip`
   - `v6p4_clean_reports.zip`
3. Tải parent V6.2E từ Camber và reconstruct frozen `train/val/test`.
4. Kiểm tra dataset integrity.
5. Evaluate checkpoint V6.4 trên **RAW VAL**.
6. Tính AP50 / AP75 / AP90 / AP95 / mAP50:95.
7. Tách residual thành:
   - detection miss,
   - center localization error,
   - bbox extent/shape error.
8. Xuất contact sheets GT đỏ / prediction xanh.

## Camber roots chính thức

Dataset bundle:

`stash://bunpmc/projects/yolov11m/datasets/v6p4_clean/v6p4_clean_official`

Checkpoint:

`stash://bunpmc/projects/yolov11m/checkpoints/v6p4_clean/v6p4_clean_official/yolo11m_v6p4_clean_official_best.pt`

### Giới hạn provenance quan trọng

Bundle V6.4 hiện tại không chứa exact training runtime với đầy đủ 79 missing-GT edits + 600 hard crops.
Vì vậy notebook này **không giả vờ reconstruct exact V6.4 train set**.

RAW VAL của V6.4 được freeze từ V6.2E, nên vẫn có thể đánh giá checkpoint và phân tích localization một cách hợp lệ trên VAL.

In [14]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import hashlib
import random

PROJECT = Path("/mnt/c/Users/intern/Documents/SEP490/train")

W = PROJECT
BOOT = W / "data" / "v6p4_camber_bootstrap"
TMP = W / "data" / "v6p4_clean_pipeline"

RUNTIME = BOOT / "v6p2e_runtime"
OUT = W / "artifacts" / "v6p4_post_clean_audit"

for p in [BOOT, TMP, RUNTIME, OUT]:
    p.mkdir(parents=True, exist_ok=True)

print("Workspace:", W)
print("BOOT:", BOOT)
print("RUNTIME:", RUNTIME)

Workspace: /mnt/c/Users/intern/Documents/SEP490/train
BOOT: /mnt/c/Users/intern/Documents/SEP490/train/data/v6p4_camber_bootstrap
RUNTIME: /mnt/c/Users/intern/Documents/SEP490/train/data/v6p4_camber_bootstrap/v6p2e_runtime


In [15]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT = Path("/mnt/c/Users/intern/Documents/SEP490/train")

W = PROJECT
BOOT = W / "data" / "v6p4_camber_bootstrap"
TMP = W / "data" / "v6p4_clean_pipeline"
RUNTIME = BOOT / "v6p2e_runtime"
OUT = W / "artifacts" / "v6p4_post_clean_audit"

# Remote Camber artifact
V6P4_DATA_ROOT = (
    "stash://bunpmc/projects/yolov11m/datasets/"
    "v6p4_clean/v6p4_clean_official"
)

LOCAL_V6P4 = W / "data" / "v6p4_clean_official"
LOCAL_V6P4.mkdir(parents=True, exist_ok=True)

for p in [BOOT, TMP, RUNTIME, OUT]:
    p.mkdir(parents=True, exist_ok=True)

CAMBER = shutil.which("camber") or str(Path.home() / ".camber/bin/camber")

def stash_get(remote, local):
    local = Path(local)
    local.parent.mkdir(parents=True, exist_ok=True)

    result = subprocess.run(
        [CAMBER, "stash", "cp", remote, str(local)],
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )

    if result.returncode != 0 or not local.exists():
        raise RuntimeError(result.stderr or result.stdout)

    print(f"Downloaded: {local}")

print("Local V6.4 configuration loaded")
print("Project:", PROJECT)
print("Camber:", CAMBER)

Local V6.4 configuration loaded
Project: /mnt/c/Users/intern/Documents/SEP490/train
Camber: /home/intern/.camber/bin/camber


In [16]:
# ==============================================================================
# 1. INSTALL + CAMBER AUTH
# ==============================================================================
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.4.172", "pyyaml", "pillow",
    "pandas", "imagehash", "tqdm"
])

import yaml
import imagehash
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from ultralytics import YOLO

# Kaggle secret
key = os.environ.get("CAMBER_API_KEY")
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("CAMBER_API_KEY")
    except Exception:
        key = None

if not key:
    raise RuntimeError(
        "Không tìm thấy CAMBER_API_KEY. "
        "Bật secret CAMBER_API_KEY trong Kaggle Add-ons -> Secrets."
    )

os.environ["CAMBER_API_KEY"] = key

camber_bin = Path.home() / ".camber/bin/camber"
if shutil.which("camber") is None and not camber_bin.exists():
    print("Installing Camber CLI...")
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True,
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:"
    + os.environ.get("PATH", "")
)
CAMBER = shutil.which("camber") or str(camber_bin)

if not Path(CAMBER).exists():
    raise RuntimeError("Camber CLI không tồn tại sau khi cài.")

print("CAMBER:", CAMBER)
print("CAMBER_API_KEY: OK")

CAMBER: /home/intern/.camber/bin/camber
CAMBER_API_KEY: OK


In [17]:
# ==============================================================================
# 2. CAMBER HELPERS
# ==============================================================================
def stash_cp(remote, local, reuse=True):
    local = Path(local)
    local.parent.mkdir(parents=True, exist_ok=True)

    if reuse and local.exists() and local.stat().st_size > 0:
        print("reuse:", local.name)
        return local

    cmd = [CAMBER, "stash", "cp", str(remote), str(local)]
    r = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )

    if r.returncode != 0 or not local.exists():
        raise RuntimeError(
            f"Camber cp failed\nREMOTE: {remote}\nLOCAL: {local}\n"
            f"{r.stderr or r.stdout}"
        )

    print(f"downloaded: {remote} -> {local}")
    return local

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            b = f.read(1024 * 1024)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

In [18]:
# Remote Camber root
V6P4_DATA_ROOT = (
    "stash://bunpmc/projects/yolov11m/datasets/"
    "v6p4_clean/v6p4_clean_official"
)

V6P4_CKPT = (
    "stash://bunpmc/projects/yolov11m/checkpoints/"
    "v6p4_clean/v6p4_clean_official/"
    "yolo11m_v6p4_clean_official_best.pt"
)

# Local destination
OFFICIAL_DIR = PROJECT / "data" / "v6p4_clean_official"
CACHE = PROJECT / "data" / "cache"
OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

artifacts = {
    "best.pt": f"{V6P4_DATA_ROOT}/best.pt",
    "data.yaml": f"{V6P4_DATA_ROOT}/data.yaml",
    "v6p4_clean_official.freeze.json":
        f"{V6P4_DATA_ROOT}/v6p4_clean_official.freeze.json",
    "v6p4_clean_reconciled_labels.zip":
        f"{V6P4_DATA_ROOT}/v6p4_clean_reconciled_labels.zip",
    "v6p4_clean_reports.zip":
        f"{V6P4_DATA_ROOT}/v6p4_clean_reports.zip",
}

downloaded = {}

for name, remote in artifacts.items():
    local_path = OFFICIAL_DIR / name
    stash_cp(remote, local_path)
    downloaded[name] = local_path

ckpt_copy = CACHE / "yolo11m_v6p4_clean_official_best.pt"
stash_cp(V6P4_CKPT, ckpt_copy)

WEIGHTS = OFFICIAL_DIR / "best.pt"
DATA_YAML_OFFICIAL = OFFICIAL_DIR / "data.yaml"
FREEZE_JSON = OFFICIAL_DIR / "v6p4_clean_official.freeze.json"
LABELS_ZIP = OFFICIAL_DIR / "v6p4_clean_reconciled_labels.zip"
REPORTS_ZIP = OFFICIAL_DIR / "v6p4_clean_reports.zip"

print("Downloaded V6.4 bundle:")
for p in [WEIGHTS, DATA_YAML_OFFICIAL, FREEZE_JSON, LABELS_ZIP, REPORTS_ZIP]:
    print(f"{p.name}: {p.stat().st_size / 1024**2:.2f} MB")

same_ckpt = sha256_file(WEIGHTS) == sha256_file(ckpt_copy)
print("Checkpoint copies identical:", same_ckpt)
assert same_ckpt

official_freeze = json.loads(FREEZE_JSON.read_text())
official_cfg = yaml.safe_load(DATA_YAML_OFFICIAL.read_text())

print(json.dumps(official_freeze, indent=2))
print(DATA_YAML_OFFICIAL.read_text())

reuse: best.pt
reuse: data.yaml
reuse: v6p4_clean_official.freeze.json
reuse: v6p4_clean_reconciled_labels.zip
reuse: v6p4_clean_reports.zip
reuse: yolo11m_v6p4_clean_official_best.pt
Downloaded V6.4 bundle:
best.pt: 38.69 MB
data.yaml: 0.00 MB
v6p4_clean_official.freeze.json: 0.00 MB
v6p4_clean_reconciled_labels.zip: 0.03 MB
v6p4_clean_reports.zip: 6.62 MB
Checkpoint copies identical: True
{
  "schema_version": "v6p4-data-centric-freeze-clean-1",
  "created_utc": "2026-10-07T09:45:54.668633+00:00",
  "dataset_name": "V6.4-clean",
  "final_tag": "v6p4_clean_official",
  "parent_tag": "v6p2e_23562a45b343",
  "split_counts": {
    "train": 17483,
    "val": 2272,
    "test": 2012
  },
  "label_modifications": {
    "missing_gt_added": 79,
    "human_verified_bird_nests": 79,
    "false_alarm_excluded": "Case 3",
    "broken_strand_policy": "Defect-Level Cluster"
  },
  "artifacts": {
    "labels_zip": "v6p4_clean_reconciled_labels.zip",
    "reports_zip": "v6p4_clean_reports.zip",
    "w

In [19]:
# ==============================================================================
# 4. AUDIT OFFICIAL V6.4 ARTIFACT BUNDLE
# ==============================================================================
import json
import zipfile
import hashlib
from pathlib import Path

import yaml

with zipfile.ZipFile(LABELS_ZIP) as z:
    label_members = [n for n in z.namelist() if n.lower().endswith(".txt")]

with zipfile.ZipFile(REPORTS_ZIP) as z:
    report_members = z.namelist()

artifact_audit = {
    "v6p4_data_root": V6P4_DATA_ROOT,
    "v6p4_checkpoint_root": V6P4_CKPT,
    "checkpoint_sha256": sha256_file(WEIGHTS),
    "checkpoint_copies_identical": same_ckpt,
    "official_yaml_path": official_cfg.get("path"),
    "official_freeze": official_freeze,
    "labels_zip_txt_count": len(label_members),
    "reports_zip_member_count": len(report_members),
    "exact_v6p4_train_runtime_packaged": False,
    "reason": (
        "Current official bundle contains checkpoint/metadata/label archive/reports, "
        "but not the image runtime nor an exact replay package for all training mutations "
        "(79 missing-GT edits + 600 hard crops)."
    ),
}

(OUT / "official_artifact_audit.json").write_text(
    json.dumps(artifact_audit, indent=2)
)

print(json.dumps(artifact_audit, indent=2))

{
  "v6p4_data_root": "stash://bunpmc/projects/yolov11m/datasets/v6p4_clean/v6p4_clean_official",
  "v6p4_checkpoint_root": "stash://bunpmc/projects/yolov11m/checkpoints/v6p4_clean/v6p4_clean_official/yolo11m_v6p4_clean_official_best.pt",
  "checkpoint_sha256": "a6e34cdbab6e8dbf8565c520dcc7bcc835c11ecca0009d7c5cd34d0abff7ca52",
  "checkpoint_copies_identical": true,
  "official_yaml_path": "/tmp/v6p4_clean_runtime",
  "official_freeze": {
    "schema_version": "v6p4-data-centric-freeze-clean-1",
    "created_utc": "2026-10-07T09:45:54.668633+00:00",
    "dataset_name": "V6.4-clean",
    "final_tag": "v6p4_clean_official",
    "parent_tag": "v6p2e_23562a45b343",
    "split_counts": {
      "train": 17483,
      "val": 2272,
      "test": 2012
    },
    "label_modifications": {
      "missing_gt_added": 79,
      "human_verified_bird_nests": 79,
      "false_alarm_excluded": "Case 3",
      "broken_strand_policy": "Defect-Level Cluster"
    },
    "artifacts": {
      "labels_zip": "v6p

In [20]:
# ==============================================================================
# 5. DOWNLOAD V6.2E PARENT ARTIFACTS FROM CAMBER
# ==============================================================================
V6P2E_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
V6P2E_TAG = "v6p2e_23562a45b343"

CACHE = PROJECT / "data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

v6p2e_freeze_path = CACHE / "v6p2e.freeze.json"
stash_cp(
    f"{V6P2E_ROOT}/v6p2e/{V6P2E_TAG}/v6p2e.freeze.json",
    v6p2e_freeze_path,
)

parent_freeze = json.loads(v6p2e_freeze_path.read_text())
print("Parent freeze:", json.dumps(parent_freeze, indent=2))

base_tar = CACHE / f"base_{parent_freeze['base_tag']}.tar"
overlay_tar = CACHE / f"overlay_{parent_freeze['overlay_tag']}.tar.gz"
parent_labels_zip = CACHE / "v6p2e_reconciled_labels.zip"
manifest_csv = CACHE / "v6p2e_split_manifest.csv"

stash_cp(
    f"{V6P2E_ROOT}/curated_v6_rs1280_{parent_freeze['base_tag']}.tar",
    base_tar,
)
stash_cp(
    f"{V6P2E_ROOT}/v6p1/{parent_freeze['parent_dataset_tag']}/"
    f"v6p1_fotl_overlay_{parent_freeze['overlay_tag']}.tar.gz",
    overlay_tar,
)
stash_cp(
    f"{V6P2E_ROOT}/v6p2e/{V6P2E_TAG}/v6p2e_reconciled_labels.zip",
    parent_labels_zip,
)
stash_cp(
    f"{V6P2E_ROOT}/v6p2e/{V6P2E_TAG}/v6p2e_split_manifest.csv",
    manifest_csv,
)

reuse: v6p2e.freeze.json
Parent freeze: {
  "schema_version": "v6p2e-strict-conservative-quarantine-1",
  "created_utc": "2026-10-03T04:53:00.958147+00:00",
  "dataset_name": "V6.2E",
  "final_tag": "v6p2e_23562a45b343",
  "parent_dataset_tag": "v6p1_5d0b1149891c",
  "base_tag": "56b7ea194ee5",
  "overlay_tag": "ee2d3c4497fd",
  "taxonomy": [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand"
  ],
  "split_counts": {
    "train": 17483,
    "val": 2272,
    "test": 2012
  },
  "test_policy": "exact SHA256 set preserved from V6.1; TEST never moved or removed",
  "exact_duplicate_audit": {
    "raw_trainval": 19918,
    "sha_groups": 19782,
    "resolved_records": 19782,
    "ambiguous_groups_quarantined": 0
  },
  "near_duplicate_audit": {
    "strong_edges": 1416,
    "strong_groups": 1403,
    "strong_test_related_trainval_quarantined": 8,
    "medium_test_candidate_pairs": 19,
    "medium_test_related_unique_

PosixPath('/mnt/c/Users/intern/Documents/SEP490/train/data/cache/v6p2e_split_manifest.csv')

In [21]:
import tarfile
import zipfile
import yaml
from pathlib import Path
from tqdm.auto import tqdm

NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

BASE_EXTRACT = CACHE / "base_extract"
OVERLAY_EXTRACT = CACHE / "overlay_extract"
RECON_LABELS = CACHE / "reconciled_labels"

for folder in [BASE_EXTRACT, OVERLAY_EXTRACT, RECON_LABELS]:
    folder.mkdir(parents=True, exist_ok=True)


def norm_names(names):
    if isinstance(names, dict):
        return [
            names[k]
            for k in sorted(names, key=lambda x: int(x))
        ]
    return list(names or [])


def find_sixclass_root(root):
    problems = []

    for yp in sorted(Path(root).rglob("data.yaml")):
        try:
            cfg = yaml.safe_load(
                yp.read_text(encoding="utf-8")
            ) or {}

            names = norm_names(cfg.get("names", []))

            if names == NAMES:
                print("Dataset root:", yp.parent)
                return yp.parent

            problems.append(f"{yp}: unexpected names={names}")

        except (OSError, ValueError, TypeError, yaml.YAMLError) as exc:
            problems.append(f"{yp}: {exc}")

    details = "\n".join(problems) or "No data.yaml files found"
    raise RuntimeError(
        f"Không tìm thấy six-class dataset root dưới {root}:\n"
        f"{details}"
    )

In [22]:
import numpy as np
import pandas as pd

In [23]:
# ==============================================================================
# 7. LABEL / GEOMETRY DATASET GATE
# ==============================================================================
import numpy as np 
import pandas as pd

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def parse_label(lp):
    rows, errs = [], []

    if not lp.exists():
        return rows, errs, "missing"

    txt = lp.read_text(errors="ignore").strip()
    if not txt:
        return rows, errs, "empty"

    for line_no, line in enumerate(txt.splitlines(), 1):
        p = line.strip().split()

        if len(p) < 5:
            errs.append((line_no, "too_few_fields", line))
            continue

        try:
            cid = int(float(p[0]))
            xc, yc, w, h = map(float, p[1:5])
        except Exception:
            errs.append((line_no, "parse_error", line))
            continue

        if cid < 0 or cid >= len(NAMES):
            errs.append((line_no, "bad_class", line))

        if not all(np.isfinite([xc, yc, w, h])):
            errs.append((line_no, "non_finite", line))

        if w <= 0 or h <= 0:
            errs.append((line_no, "non_positive_wh", line))

        x1, y1 = xc-w/2, yc-h/2
        x2, y2 = xc+w/2, yc+h/2
        overflow = max(0.0, -x1, -y1, x2-1.0, y2-1.0)

        rows.append({
            "cid": cid,
            "class": NAMES[cid] if 0 <= cid < len(NAMES) else str(cid),
            "xc": xc, "yc": yc, "w": w, "h": h,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "overflow": overflow,
        })

    return rows, errs, "ok"

box_rows = []
err_rows = []
image_rows = []

for split in ["train", "val", "test"]:
    image_dir = RUNTIME / "images" / split
    label_dir = RUNTIME / "labels" / split

    images = sorted(
        p for p in image_dir.rglob("*")
        if p.suffix.lower() in IMG_EXTS
    )

    for ip in tqdm(images, desc=f"label scan {split}"):
        lp = label_dir / ip.relative_to(image_dir).with_suffix(".txt")
        rows, errs, status = parse_label(lp)

        image_rows.append({
            "split": split,
            "image": str(ip),
            "label_status": status,
            "n_boxes": len(rows),
        })

        for e in errs:
            err_rows.append({
                "split": split,
                "image": str(ip),
                "label": str(lp),
                "line": e[0],
                "error": e[1],
                "raw": e[2],
            })

        for j, r in enumerate(rows):
            rr = dict(r)
            rr.update({
                "split": split,
                "image": str(ip),
                "label": str(lp),
                "gt_index": j,
            })
            box_rows.append(rr)

df_boxes = pd.DataFrame(box_rows)
df_err = pd.DataFrame(err_rows)
df_img = pd.DataFrame(image_rows)

print("\nImage/label status:")
print(
    df_img.groupby(["split", "label_status"])
    .size()
    .unstack(fill_value=0)
)

print("\nBoxes:")
print(pd.crosstab(df_boxes["split"], df_boxes["class"]))

print("\nLabel errors:", len(df_err))
if len(df_err):
    display(df_err.head(100))

df_boxes["area"] = df_boxes["w"] * df_boxes["h"]
df_boxes["aspect"] = (
    df_boxes["w"] / df_boxes["h"].clip(lower=1e-12)
)
df_boxes["edge_touch"] = (
    (df_boxes["x1"] <= 1e-6) |
    (df_boxes["y1"] <= 1e-6) |
    (df_boxes["x2"] >= 1-1e-6) |
    (df_boxes["y2"] >= 1-1e-6)
)

geom = (
    df_boxes.groupby(["split", "class"])
    .agg(
        boxes=("area", "size"),
        median_area=("area", "median"),
        p10_area=("area", lambda x: x.quantile(.10)),
        p90_area=("area", lambda x: x.quantile(.90)),
        median_aspect=("aspect", "median"),
        edge_touch_rate=("edge_touch", "mean"),
        max_overflow=("overflow", "max"),
    )
    .reset_index()
)

display(geom)

df_boxes.to_csv(
    OUT / "bbox_records.csv.gz",
    index=False,
    compression="gzip",
)
df_err.to_csv(OUT / "label_errors.csv", index=False)
geom.to_csv(OUT / "bbox_geometry_profile.csv", index=False)

label scan train:   0%|          | 0/17483 [00:00<?, ?it/s]

label scan val:   0%|          | 0/2272 [00:00<?, ?it/s]

label scan test:   0%|          | 0/2012 [00:00<?, ?it/s]


Image/label status:
label_status  empty     ok
split                     
test            145   1867
train          1437  16046
val             179   2093

Boxes:
class  bird_nest  broken_strand  foreign_object  insulator  insulator_broken  \
split                                                                          
test         178            134             161       1267               809   
train       6085           2002             667       6226              6236   
val          821            252              98        778               791   

class  insulator_flashover  
split                       
test                   124  
train                 1249  
val                    163  

Label errors: 0


,split,class,boxes,median_area,p10_area,p90_area,median_aspect,edge_touch_rate,max_overflow
0,test,bird_nest,178,0.015970,0.001548,0.170570,0.996150,0.050562,1.562500e-03
1,test,broken_strand,134,0.075737,0.004820,0.304695,1.593320,0.149254,5.000000e-07
2,test,foreign_object,161,0.082468,0.009584,0.235973,0.643697,0.006211,0.000000e+00
3,test,insulator,1267,0.027214,0.000650,0.197679,0.258560,0.569061,5.000000e-07
4,test,insulator_broken,809,0.014665,0.002874,0.076918,0.994806,0.013597,5.000000e-07
5,test,insulator_flashover,124,0.019519,0.006288,0.045059,0.841809,0.000000,0.000000e+00
6,train,bird_nest,6085,0.007291,0.000647,0.070927,1.037500,0.022350,5.000000e-07
7,train,broken_strand,2002,0.085211,0.011122,0.294688,2.163473,0.139860,5.000000e-07
8,train,foreign_object,667,0.081133,0.009110,0.268624,0.661631,0.014993,0.000000e+00
9,train,insulator,6226,0.011551,0.000477,0.135795,0.287553,0.354642,5.000000e-07


In [24]:
# ==============================================================================
# 8. CROSS-SPLIT DUPLICATE AUDIT
# ==============================================================================
DUP_GT_IOU = 0.95
RUN_PHASH = True
PHASH_MAX_HAMMING = 4

def iou(a, b):
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    iw = max(0.0, ix2-ix1); ih = max(0.0, iy2-iy1)
    inter = iw*ih

    aa = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    bb = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])

    u = aa + bb - inter
    return inter/u if u > 0 else 0.0

# Same-class duplicate GT candidates.
dup_gt = []
for (split, image), g in tqdm(
    df_boxes.groupby(["split", "image"]),
    desc="duplicate GT",
):
    rows = g.to_dict("records")
    for i in range(len(rows)):
        for j in range(i+1, len(rows)):
            if rows[i]["cid"] != rows[j]["cid"]:
                continue

            a = [rows[i]["x1"], rows[i]["y1"], rows[i]["x2"], rows[i]["y2"]]
            b = [rows[j]["x1"], rows[j]["y1"], rows[j]["x2"], rows[j]["y2"]]
            z = iou(a, b)

            if z >= DUP_GT_IOU:
                dup_gt.append({
                    "split": split,
                    "image": image,
                    "class": rows[i]["class"],
                    "gt_i": rows[i]["gt_index"],
                    "gt_j": rows[j]["gt_index"],
                    "iou": z,
                })

df_dupgt = pd.DataFrame(dup_gt)
df_dupgt.to_csv(
    OUT / "duplicate_gt_candidates.csv",
    index=False,
)
print("Duplicate GT candidates:", len(df_dupgt))

# SHA256 exact cross-split.
hash_rows = []
for split in ["train", "val", "test"]:
    for ip in tqdm(
        sorted((RUNTIME / "images" / split).glob("*")),
        desc=f"SHA {split}",
    ):
        if ip.suffix.lower() not in IMG_EXTS:
            continue

        hash_rows.append({
            "split": split,
            "image": str(ip),
            "sha256": sha256_file(ip),
        })

df_hash = pd.DataFrame(hash_rows)

exact_cross = []
for sha, g in df_hash.groupby("sha256"):
    splits = sorted(set(g["split"]))
    if len(splits) > 1:
        exact_cross.append({
            "sha256": sha,
            "splits": ",".join(splits),
            "images": " | ".join(g["image"].tolist()),
        })

df_exact = pd.DataFrame(exact_cross)
df_exact.to_csv(
    OUT / "cross_split_exact_duplicates.csv",
    index=False,
)
print("Exact cross-split duplicate groups:", len(df_exact))

# pHash candidate audit.
near_rows = []

if RUN_PHASH:
    ph_rows = []

    for split in ["train", "val", "test"]:
        for ip in tqdm(
            sorted((RUNTIME / "images" / split).glob("*")),
            desc=f"pHash {split}",
        ):
            if ip.suffix.lower() not in IMG_EXTS:
                continue

            try:
                with Image.open(ip) as im:
                    ph = imagehash.phash(
                        im.convert("RGB"),
                        hash_size=8,
                    )
                ph_rows.append({
                    "split": split,
                    "image": str(ip),
                    "phash_int": int(str(ph), 16),
                })
            except Exception:
                pass

    df_ph = pd.DataFrame(ph_rows)

    widths = [13, 13, 13, 13, 12]
    shifts = []
    cur = 0
    for w in widths:
        shifts.append(cur)
        cur += w

    buckets = {}

    for idx, r in df_ph.iterrows():
        v = int(r["phash_int"])

        for band, (w, sh) in enumerate(zip(widths, shifts)):
            key = (
                band,
                (v >> sh) & ((1 << w) - 1),
            )
            buckets.setdefault(key, []).append(idx)

    candidate_pairs = set()

    for ids in buckets.values():
        if len(ids) < 2:
            continue

        for ai in range(len(ids)):
            for bi in range(ai+1, len(ids)):
                a, b = ids[ai], ids[bi]

                if df_ph.at[a, "split"] == df_ph.at[b, "split"]:
                    continue

                candidate_pairs.add(
                    (min(a, b), max(a, b))
                )

    for a, b in tqdm(
        candidate_pairs,
        desc="pHash compare",
    ):
        va = int(df_ph.at[a, "phash_int"])
        vb = int(df_ph.at[b, "phash_int"])
        hd = (va ^ vb).bit_count()

        if hd <= PHASH_MAX_HAMMING:
            near_rows.append({
                "split_a": df_ph.at[a, "split"],
                "image_a": df_ph.at[a, "image"],
                "split_b": df_ph.at[b, "split"],
                "image_b": df_ph.at[b, "image"],
                "phash_hamming": hd,
            })

df_near = pd.DataFrame(near_rows)
df_near.to_csv(
    OUT / "cross_split_phash_near_duplicates.csv",
    index=False,
)
print("pHash near-duplicate pairs:", len(df_near))

duplicate GT:   0%|          | 0/20006 [00:00<?, ?it/s]

Duplicate GT candidates: 0


SHA train:   0%|          | 0/17483 [00:00<?, ?it/s]

SHA val:   0%|          | 0/2272 [00:00<?, ?it/s]

SHA test:   0%|          | 0/2012 [00:00<?, ?it/s]

...

pHash train:   0%|          | 0/17483 [00:00<?, ?it/s]

pHash val:   0%|          | 0/2272 [00:00<?, ?it/s]

pHash test:   0%|          | 0/2012 [00:00<?, ?it/s]

pHash compare:   0%|          | 0/142000 [00:00<?, ?it/s]

pHash near-duplicate pairs: 135


In [28]:
# ==============================================================================
# 9. STRICT-IoU VALIDATION ON RAW VAL
# ==============================================================================
import json
import numpy as np
import pandas as pd

IMGSZ = 1024
VAL_BATCH = 8
DEVICE = 0

# RAW VAL của runtime V6.2E đã reconstruct ở local.
DATA_YAML_RAW = RUNTIME / "data.yaml"

if not DATA_YAML_RAW.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_YAML_RAW}. "
        "Cần hoàn tất bước reconstruct runtime ở cell 6."
    )

print("RAW VAL dataset:", DATA_YAML_RAW)

model = YOLO(str(WEIGHTS))

metrics = model.val(
    data=str(DATA_YAML_RAW),
    split="val",
    imgsz=IMGSZ,
    batch=VAL_BATCH,
    device=DEVICE,
    plots=False,
    verbose=False,
)

m = metrics.box

all_ap = np.asarray(m.all_ap, dtype=float)
ap50 = np.asarray(m.ap50, dtype=float)
class_ids = [
    int(x)
    for x in np.asarray(m.ap_class_index).tolist()
]

if all_ap.ndim == 1:
    all_ap = all_ap[:, None]

thresholds = 0.50 + 0.05 * np.arange(all_ap.shape[1])

def col_for(t):
    return int(np.argmin(np.abs(thresholds - t)))

per_class = []

for row_i, cid in enumerate(class_ids):
    vals = all_ap[row_i]

    row = {
        "cid": cid,
        "class": NAMES[cid],
        "AP50": float(ap50[row_i]),
        "AP50_95": float(np.mean(vals)),
        "gap_50_to_50_95": float(ap50[row_i] - np.mean(vals)),
    }

    for t in [0.75, 0.90, 0.95]:
        col = col_for(t)
        row[f"AP{int(round(t * 100))}"] = (
            float(vals[col])
            if np.isclose(thresholds[col], t)
            else np.nan
        )

    per_class.append(row)

df_strict = pd.DataFrame(per_class)
display(df_strict)

strict_report = {
    "mAP50": float(m.map50),
    "mAP50_95": float(m.map),
    "gap": float(m.map50 - m.map),
    "precision": float(m.mp),
    "recall": float(m.mr),
    "target_gap_le_0p30": bool((m.map50 - m.map) <= 0.30),
    "per_class": per_class,
}

print(json.dumps(strict_report, indent=2))

OUT.mkdir(parents=True, exist_ok=True)

df_strict.to_csv(
    OUT / "strict_iou_per_class.csv",
    index=False,
)

(OUT / "strict_iou_report.json").write_text(
    json.dumps(strict_report, indent=2),
    encoding="utf-8",
)

RAW VAL dataset: /mnt/c/Users/intern/Documents/SEP490/train/data/v6p4_camber_bootstrap/v6p2e_runtime/data.yaml
Ultralytics 8.4.172 🚀 Python-3.14.4 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
YOLO11m summary (fused): 125 layers, 20,034,658 parameters, 0 gradients, 67.8 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 2.4±0.2 ms, read: 36.0±39.6 MB/s, size: 141.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /mnt/c/Users/intern/Documents/SEP490/train/data/v6p4_camber_bootstrap/v6p2e_runtime/labels/val... 2272 images, 179 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2272/2272 672.7it/s 3.4s0.1s
val: New cache created: /mnt/c/Users/intern/Documents/SEP490/train/data/v6p4_camber_bootstrap/v6p2e_runtime/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 80% ━━━━━━━━━╸── 230/284 4.5s/it 12:21<4:051

[W1008 22:45:06.630708811 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 574619648 bytes (free: 0, total: 6441926656).


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 284/284 2.7s/it 12:580.4ss
                   all       2272       2903      0.877       0.78      0.839      0.472
Speed: 2.0ms preprocess, 285.2ms inference, 0.0ms loss, 1.8ms postprocess per image


,cid,class,AP50,AP50_95,gap_50_to_50_95,AP75,AP90,AP95
0,0,insulator,0.976839,0.708565,0.268275,0.803465,0.299102,0.035678
1,1,insulator_broken,0.873096,0.459860,0.413236,0.429036,0.027026,0.002146
2,2,insulator_flashover,0.858390,0.377620,0.480769,0.245899,0.009848,0.000000
3,3,bird_nest,0.880136,0.540124,0.340013,0.596687,0.047608,0.000869
4,4,foreign_object,0.655955,0.376107,0.279848,0.336642,0.102714,0.037500
5,5,broken_strand,0.788015,0.367977,0.420037,0.269915,0.004104,0.000042


{
  "mAP50": 0.8387384540070597,
  "mAP50_95": 0.47170881908590145,
  "gap": 0.36702963492115825,
  "precision": 0.8773129222981528,
  "recall": 0.7798219733752009,
  "target_gap_le_0p30": false,
  "per_class": [
    {
      "cid": 0,
      "class": "insulator",
      "AP50": 0.9768393644536388,
      "AP50_95": 0.7085647747134799,
      "gap_50_to_50_95": 0.26827458974015894,
      "AP75": 0.8034647072997956,
      "AP90": 0.2991024267337748,
      "AP95": 0.03567847852750731
    },
    {
      "cid": 1,
      "class": "insulator_broken",
      "AP50": 0.8730958679565921,
      "AP50_95": 0.4598596946906001,
      "gap_50_to_50_95": 0.413236173265992,
      "AP75": 0.4290364225052101,
      "AP90": 0.027026318598268536,
      "AP95": 0.0021457085828343312
    },
    {
      "cid": 2,
      "class": "insulator_flashover",
      "AP50": 0.8583897662518872,
      "AP50_95": 0.37762027846960533,
      "gap_50_to_50_95": 0.4807694877822819,
      "AP75": 0.24589883678008856,
      "AP90": 

1892

In [ ]:
# ==============================================================================
# 10. RESIDUAL LOCALIZATION DECOMPOSITION ON VAL
# ==============================================================================

RESIDUAL_BATCH = 8
RESIDUAL_CONF = 0.10
GOOD_IOU = 0.75
CENTER_MATCH_MAX = 0.25

def center(box):
    return np.array([
        (box[0]+box[2])/2,
        (box[1]+box[3])/2,
    ], dtype=float)

def wh(box):
    return np.array([
        max(0.0, box[2]-box[0]),
        max(0.0, box[3]-box[1]),
    ], dtype=float)

def center_dist(a, b):
    return float(np.linalg.norm(center(a)-center(b)))

def centered_pred_box(pred_box, gt_box):
    c = center(gt_box)
    w, h = wh(pred_box)

    return np.array([
        c[0]-w/2,
        c[1]-h/2,
        c[0]+w/2,
        c[1]+h/2,
    ], dtype=float)

def area(box):
    w, h = wh(box)
    return float(w*h)

def greedy_match(gts, preds):
    pairs = []

    for gi, g in enumerate(gts):
        for pi, p in enumerate(preds):
            if g["cid"] != p["cid"]:
                continue

            z = iou(g["xyxy"], p["xyxy"])
            cd = center_dist(g["xyxy"], p["xyxy"])

            if cd > CENTER_MATCH_MAX and z < 0.05:
                continue

            pairs.append(
                (z - 0.25*cd, gi, pi, z, cd)
            )

    pairs.sort(reverse=True)

    used_g, used_p = set(), set()
    matches = {}

    for score, gi, pi, z, cd in pairs:
        if gi in used_g or pi in used_p:
            continue

        used_g.add(gi)
        used_p.add(pi)
        matches[gi] = (pi, z, cd)

    return matches

val_image_dir = RUNTIME / "images" / "val"
val_label_dir = RUNTIME / "labels" / "val"
val_images = sorted(
    p for p in val_image_dir.glob("*")
    if p.suffix.lower() in IMG_EXTS
)

residual_rows = []

for start in tqdm(
    range(0, len(val_images), RESIDUAL_BATCH),
    desc="residual batches",
):
    batch_paths = val_images[
        start:start+RESIDUAL_BATCH
    ]

    results = model.predict(
        source=[str(p) for p in batch_paths],
        imgsz=IMGSZ,
        conf=RESIDUAL_CONF,
        device=DEVICE,
        verbose=False,
    )

    for ip, res in zip(batch_paths, results):
        lp = val_label_dir / ip.with_suffix(".txt").name
        gt_raw, _, _ = parse_label(lp)

        gts = []
        for j, g in enumerate(gt_raw):
            gts.append({
                "gt_index": j,
                "cid": int(g["cid"]),
                "class": g["class"],
                "xyxy": np.array([
                    g["x1"], g["y1"],
                    g["x2"], g["y2"],
                ], dtype=float),
            })

        preds = []

        if res.boxes is not None and len(res.boxes):
            cids = (
                res.boxes.cls.detach().cpu()
                .numpy().astype(int)
            )
            confs = (
                res.boxes.conf.detach().cpu()
                .numpy()
            )
            boxes = (
                res.boxes.xyxyn.detach().cpu()
                .numpy()
            )

            for j in range(len(cids)):
                preds.append({
                    "pred_index": j,
                    "cid": int(cids[j]),
                    "conf": float(confs[j]),
                    "xyxy": boxes[j].astype(float),
                })

        matches = greedy_match(gts, preds)

        for gi, g in enumerate(gts):
            gb = g["xyxy"]

            if gi not in matches:
                residual_rows.append({
                    "image": str(ip),
                    "gt_index": gi,
                    "class": g["class"],
                    "status": "DETECTION_MISS",
                    "pred_conf": None,
                    "actual_iou": 0.0,
                    "centered_iou": 0.0,
                    "center_dist": None,
                    "area_ratio": None,
                    "gt_xyxy": gb.tolist(),
                    "pred_xyxy": None,
                })
                continue

            pi, z, cd = matches[gi]
            p = preds[pi]
            pb = p["xyxy"]

            centered = centered_pred_box(pb, gb)
            cz = iou(gb, centered)

            if z >= GOOD_IOU:
                status = "GOOD75"
            elif cz >= GOOD_IOU:
                status = "CENTER_ERROR"
            else:
                status = "EXTENT_SHAPE_ERROR"

            residual_rows.append({
                "image": str(ip),
                "gt_index": gi,
                "class": g["class"],
                "status": status,
                "pred_conf": p["conf"],
                "actual_iou": float(z),
                "centered_iou": float(cz),
                "center_dist": float(cd),
                "area_ratio":
                    float(area(pb) / max(area(gb), 1e-12)),
                "gt_xyxy": gb.tolist(),
                "pred_xyxy": pb.tolist(),
            })

df_res = pd.DataFrame(residual_rows)

diag_rows = []

for cls, g in df_res.groupby("class"):
    matched = g[g["status"] != "DETECTION_MISS"]

    diag_rows.append({
        "class": cls,
        "gt": len(g),
        "miss_rate":
            float((g["status"] == "DETECTION_MISS").mean()),
        "actual_pass75":
            float((g["actual_iou"] >= 0.75).mean()),
        "centered_pass75":
            float((g["centered_iou"] >= 0.75).mean()),
        "center_uplift75":
            float(
                (g["centered_iou"] >= 0.75).mean()
                - (g["actual_iou"] >= 0.75).mean()
            ),
        "extent_shape_error_rate":
            float(
                (g["status"] == "EXTENT_SHAPE_ERROR").mean()
            ),
        "median_area_ratio_matched":
            float(matched["area_ratio"].median())
            if len(matched) else np.nan,
    })

df_diag = (
    pd.DataFrame(diag_rows)
    .sort_values("actual_pass75")
)

display(df_diag)

df_res.to_csv(
    OUT / "val_residual_localization.csv.gz",
    index=False,
    compression="gzip",
)
df_diag.to_csv(
    OUT / "val_residual_class_diagnostics.csv",
    index=False,
)

NameError: name 'RESIDUAL_BATCH' is not defined

In [ ]:
# ==============================================================================
# 11. CONTACT SHEETS
# ==============================================================================

CONTACT_DIR = OUT / "contact_sheets"
CONTACT_DIR.mkdir(parents=True, exist_ok=True)

try:
    font = ImageFont.truetype(
        "DejaVuSans.ttf", 17
    )
    small = ImageFont.truetype(
        "DejaVuSans.ttf", 14
    )
except Exception:
    font = ImageFont.load_default()
    small = font

def draw_norm_box(
    draw, box, W, H, offset_y,
    outline, text,
):
    x1, y1, x2, y2 = box

    rect = [
        x1*W,
        y1*H+offset_y,
        x2*W,
        y2*H+offset_y,
    ]

    draw.rectangle(
        rect,
        outline=outline,
        width=4,
    )

    draw.text(
        (
            rect[0]+3,
            max(offset_y, rect[1]-18),
        ),
        text,
        fill=outline,
        font=small,
    )

hard = df_res[
    df_res["status"].isin([
        "CENTER_ERROR",
        "EXTENT_SHAPE_ERROR",
        "DETECTION_MISS",
    ])
].copy()

hard["_priority"] = (
    (hard["status"] == "EXTENT_SHAPE_ERROR").astype(int)*3
    + (hard["status"] == "CENTER_ERROR").astype(int)*2
    + hard["pred_conf"].fillna(0)
)

hard = hard.sort_values(
    "_priority",
    ascending=False,
).head(MAX_CONTACT_SHEETS)

for _, r in hard.iterrows():
    ip = Path(r["image"])

    try:
        im = Image.open(ip).convert("RGB")
    except Exception:
        continue

    scale = min(
        1.0,
        1100 / max(im.size),
    )

    if scale < 1:
        im = im.resize((
            int(im.width*scale),
            int(im.height*scale),
        ))

    W2, H2 = im.size
    head = 90

    canvas = Image.new(
        "RGB",
        (W2, H2+head),
        "white",
    )
    canvas.paste(im, (0, head))

    d = ImageDraw.Draw(canvas)

    d.text(
        (8, 8),
        (
            f"{r['class']} | {r['status']} | "
            f"IoU={r['actual_iou']:.3f} | "
            f"centered={r['centered_iou']:.3f}"
        ),
        fill="black",
        font=font,
    )

    d.text(
        (8, 38),
        (
            f"GT=red | PRED=green | "
            f"conf={r['pred_conf']} | "
            f"area_ratio={r['area_ratio']}"
        ),
        fill="black",
        font=small,
    )

    gt = r["gt_xyxy"]
    if isinstance(gt, str):
        gt = json.loads(
            gt.replace("'", '"')
        )

    draw_norm_box(
        d, gt, W2, H2,
        head, "red", "GT",
    )

    pb = r["pred_xyxy"]

    if pb is not None and not (
        isinstance(pb, float)
        and np.isnan(pb)
    ):
        if isinstance(pb, str):
            pb = json.loads(
                pb.replace("'", '"')
            )

        draw_norm_box(
            d, pb, W2, H2,
            head, "green", "PRED",
        )

    name = (
        f"{r['class']}_{r['status']}_"
        f"{ip.stem}_{int(r['gt_index'])}.jpg"
    )

    canvas.save(
        CONTACT_DIR / name,
        quality=90,
    )

print("Contact sheets:", CONTACT_DIR)

NameError: name 'df_res' is not defined

In [ ]:
# ==============================================================================
# 12. FINAL DECISION REPORT
# ==============================================================================
decision = {
    "artifact_audit": artifact_audit,
    "strict_iou": strict_report,
    "parent_dataset_gate": {
        "label_errors": int(len(df_err)),
        "duplicate_gt_candidates": int(len(df_dupgt)),
        "cross_split_exact_duplicate_groups":
            int(len(df_exact)),
        "cross_split_phash_pairs":
            int(len(df_near)),
    },
    "next_action": [],
    "notes": [],
}

if len(df_err) > 0 or len(df_exact) > 0:
    decision["next_action"].append(
        "FIX_PARENT_DATASET_INTEGRITY"
    )

if strict_report["gap"] <= 0.30:
    decision["next_action"].append(
        "GAP_TARGET_MET_ON_RAW_VAL"
    )
else:
    decision["next_action"].append(
        "STRICT_LOCALIZATION_GAP_STILL_HIGH"
    )

for _, r in df_diag.iterrows():
    cls = r["class"]

    if r["miss_rate"] >= 0.20:
        decision["next_action"].append(
            f"{cls}:DETECTION_COVERAGE"
        )

    if r["center_uplift75"] >= 0.10:
        decision["next_action"].append(
            f"{cls}:CENTER_LOCALIZATION"
        )

    if r["extent_shape_error_rate"] >= 0.20:
        decision["next_action"].append(
            f"{cls}:EXTENT_SHAPE_REVIEW"
        )

decision["notes"].append(
    "Không dùng reconstructed parent TRAIN để tuyên bố exact V6.4 train audit."
)

decision["notes"].append(
    "RAW VAL evaluation hợp lệ cho checkpoint vì VAL được freeze từ V6.2E."
)

seen = set()
decision["next_action"] = [
    x for x in decision["next_action"]
    if not (x in seen or seen.add(x))
]

report = OUT / "v6p4_post_clean_decision.json"
report.write_text(
    json.dumps(
        decision,
        indent=2,
        default=str,
    )
)

print(json.dumps(
    decision,
    indent=2,
    default=str,
))

print("\nImportant outputs:")
for p in [
    OUT / "official_artifact_audit.json",
    OUT / "bbox_geometry_profile.csv",
    OUT / "strict_iou_per_class.csv",
    OUT / "strict_iou_report.json",
    OUT / "val_residual_class_diagnostics.csv",
    report,
]:
    print(" -", p)

{
  "artifact_audit": {
    "v6p4_data_root": "stash://bunpmc/projects/yolov11m/datasets/v6p4_clean/v6p4_clean_official",
    "v6p4_checkpoint_root": "stash://bunpmc/projects/yolov11m/checkpoints/v6p4_clean/v6p4_clean_official/yolo11m_v6p4_clean_official_best.pt",
    "checkpoint_sha256": "a6e34cdbab6e8dbf8565c520dcc7bcc835c11ecca0009d7c5cd34d0abff7ca52",
    "checkpoint_copies_identical": true,
    "official_yaml_path": "/tmp/v6p4_clean_runtime",
    "official_freeze": {
      "schema_version": "v6p4-data-centric-freeze-clean-1",
      "created_utc": "2026-10-07T09:45:54.668633+00:00",
      "dataset_name": "V6.4-clean",
      "final_tag": "v6p4_clean_official",
      "parent_tag": "v6p2e_23562a45b343",
      "split_counts": {
        "train": 17483,
        "val": 2272,
        "test": 2012
      },
      "label_modifications": {
        "missing_gt_added": 79,
        "human_verified_bird_nests": 79,
        "false_alarm_excluded": "Case 3",
        "broken_strand_policy": "Defect-L

## Sau khi chạy xong

Gửi lại các output sau:

- `strict_iou_per_class.csv`
- `strict_iou_report.json`
- `val_residual_class_diagnostics.csv`
- `v6p4_post_clean_decision.json`

Đặc biệt quan trọng là:

- AP75 / AP90 / AP95 của `insulator_broken`, `insulator_flashover`, `broken_strand`.
- `center_uplift75`.
- `extent_shape_error_rate`.
- contact sheets của những class trên.

Từ đó mới quyết định liệu bước tiếp theo là:

- sửa annotation policy có kiểm soát,
- cải thiện coverage,
- hay chuyển sang RT-DETR/localization-side experiment.